# BNCC Machine Learning Workshop: Classification

Pada notebook ini, kita akan mempelajari alur kerja machine learning untuk masalah classification secara end-to-end. Tujuannya adalah membangun model yang dapat membedakan SMS biasa (`ham`) dan SMS spam (`spam`).

Alur notebook dibuat mirip dengan notebook regression: mulai dari memahami dataset, membersihkan data, preprocessing, training model, evaluasi, membuat pipeline, inference, sampai eksperimen model lain.


In [ ]:
print("Welcome to BNCC Machine Learning Workshop - Classification")

Sebelum membangun model, kita perlu mengetahui sumber data yang digunakan. Dataset pada notebook ini berisi kumpulan pesan SMS yang sudah diberi label `ham` atau `spam`.


Link Dataset: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

In [ ]:
import re
import string

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Memuat Dataset

Pada bagian ini, kita mulai memuat file dataset ke dalam pandas DataFrame. Setelah data berhasil dibaca, tampilkan beberapa baris pertama dan memastikan kolom-kolomnya sudah sesuai.

In [ ]:
# TODO: lengkapi kode berikut untuk memuat dataset

...

Cek struktur dataset: jumlah baris, nama kolom, tipe data, dan apakah ada missing value. Tahap ini membantu kita memastikan dataset sudah terbaca dengan benar sebelum masuk ke visualisasi dan modeling.

In [ ]:
# TODO: lengkapi kode berikut untuk melihat struktur dataset

...

In [ ]:
extra_cols = ["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]

df[df[extra_cols].notna().any(axis=1)][["v1", "v2"] + extra_cols].head()

Beberapa pesan pada dataset terpecah ke kolom `Unnamed` karena format CSV yang kurang rapi. Agar isi SMS tetap utuh, kita gabungkan kolom `v2`, `Unnamed: 2`, `Unnamed: 3`, dan `Unnamed: 4` menjadi satu kolom baru bernama `message`.

Nilai kosong diisi dengan string kosong terlebih dahulu, lalu hasil gabungan dirapikan dengan menghapus spasi berlebih.

In [ ]:
# TODO: gabungkan banyak kolom menjadi 1 kolom message

...

Informasi Fitur Dataset:

1. `label` = kelas SMS, berisi `ham` untuk SMS biasa dan `spam` untuk SMS spam.
2. `message` = isi pesan SMS dalam bentuk teks.


In [ ]:
# TODO: ambil dan rename kolom menjadi label dan message

...

Missing value perlu dicek sebelum modeling karena data kosong bisa membuat proses preprocessing atau training gagal. Hitung jumlah nilai kosong pada setiap kolom.


In [ ]:
# TODO: cek jumlah data kosong di setiap kolom

...

Data duplikat bisa membuat model melihat pesan yang sama lebih dari sekali. Cek jumlah data duplikat, lalu hapus jika ditemukan agar evaluasi model lebih adil.


In [ ]:
duplicated_data = df[df.duplicated(keep=False)].sort_values(by=["label", "message"])

duplicated_data.head()

In [ ]:
# TODO: cek dan hapus data duplikat jika ada

...

Lihat distribusi label untuk mengetahui berapa banyak SMS biasa dan SMS spam. Pada classification, distribusi kelas penting karena dataset yang tidak seimbang bisa membuat accuracy terlihat tinggi walaupun model kurang baik dalam mendeteksi kelas minoritas.


In [ ]:
# TODO: cek distribusi label ham dan spam

...

## EDA (Exploratory Data Analysis)

Visualisasi distribusi label membantu kita melihat apakah jumlah SMS `ham` dan `spam` seimbang. Jika jumlah salah satu kelas jauh lebih besar, evaluasi perlu memperhatikan precision, recall, dan F1-score, bukan hanya accuracy.


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="label", hue="label", palette="Set2", legend=False)
plt.title("Distribusi Label SMS")
plt.xlabel("Label")
plt.ylabel("Jumlah SMS")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

Panjang pesan bisa menjadi salah satu sinyal sederhana untuk membedakan spam dan ham. Pada cell ini, kita membuat fitur `message_length`, lalu membandingkan distribusinya untuk setiap label.


In [ ]:
df["message_length"] = df["message"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=df,
    x="message_length",
    hue="label",
    bins=50,
    kde=True,
    element="step",
    ax=axes[0]
)
axes[0].set_title("Distribusi Panjang Pesan")
axes[0].set_xlabel("Jumlah Karakter")
axes[0].set_ylabel("Jumlah SMS")

sns.boxplot(
    data=df,
    x="label",
    y="message_length",
    hue="label",
    palette="Set2",
    legend=False,
    ax=axes[1]
)
axes[1].set_title("Perbandingan Panjang Pesan per Label")
axes[1].set_xlabel("Label")
axes[1].set_ylabel("Jumlah Karakter")

plt.tight_layout()
plt.show()

In [ ]:
# ringkasan panjang pesan per label

message_length_summary = (
    df.groupby("label")["message_length"]
    .describe()
    .round(2)
    .reset_index()
)

message_length_summary

## Membersihkan Data

Pada tahap cleaning, kita gunakan salinan dataset agar data asli tetap aman. Isi pesan akan dinormalisasi dengan lowercase, penghapusan punctuation, dan perapihan whitespace.

Untuk target classification, kita tetap menggunakan label asli yaitu `ham` dan `spam` agar hasil evaluasi lebih mudah dibaca oleh peserta.


In [ ]:
# TODO: copy dataframe terlebih dahulu supaya dataframe asli tidak berubah

...

In [ ]:
# TODO: buat fungsi sederhana untuk membersihkan teks

...

## Stratified Train-Test Split

Sebelum membagi data, tentukan dulu mana kolom yang menjadi fitur dan mana kolom yang menjadi target. Pada kasus classification ini, fitur adalah teks pesan, sedangkan target adalah label `ham` atau `spam`.

Kita menggunakan `stratify=y` agar proporsi `ham` dan `spam` tetap mirip pada training set dan test set.


In [ ]:
# TODO: pisahkan fitur dan target

...

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: lakukan splitting dan tentukan jumlah data yang akan di jadikan testing

...

## Representasi Teks dengan TF-IDF

TF-IDF digunakan untuk mengubah teks menjadi angka agar bisa diproses oleh model machine learning. Setiap kata/token yang dipilih akan menjadi satu fitur dalam bentuk vektor.

Pada `TfidfVectorizer`, `stop_words="english"` berarti kata-kata umum dalam bahasa Inggris seperti `the`, `is`, `and`, atau `to` akan diabaikan karena biasanya tidak membawa banyak informasi untuk klasifikasi.

`max_features=3000` membatasi ukuran vocabulary maksimal menjadi 3000 token. Karena setiap token di vocabulary menjadi satu dimensi fitur, maka hasil TF-IDF akan memiliki maksimal 3000 dimensi fitur.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: ubah teks menjadi fitur numerik menggunakan TF-IDF

...

Untuk melihat token apa saja yang dipelajari oleh TF-IDF, kita bisa mengambil beberapa contoh token dari vocabulary.

Token yang muncul tidak selalu berupa kata biasa. Pada dataset SMS, token bisa berupa kata, angka, kode, atau nomor telepon karena semuanya muncul di dalam pesan. Ini membantu kita memahami bentuk fitur teks yang nantinya digunakan oleh model.

In [ ]:
# NOTE: jalankan kode ini beberapa kali untuk melihat semua kata/token yang di pelajari oleh TF-IDF

import numpy as np

feature_names = tfidf.get_feature_names_out()

np.random.choice(feature_names, size=50, replace=False)

## Training menggunakan Logistic Regression

Setelah teks selesai diubah menjadi angka, kita bisa melatih model classification. Logistic Regression sering digunakan sebagai baseline untuk classification karena sederhana, cepat, dan hasilnya mudah dibandingkan dengan model lain.


In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: lakukan training model menggunakan logistic regression

...

## Evaluasi Model Logistic Regression

Evaluasi pada training set digunakan untuk melihat seberapa baik model mempelajari data yang digunakan saat training.

Nilai ini nantinya perlu dibandingkan dengan performa pada test set. Jika performa training jauh lebih baik daripada test, model bisa jadi mengalami overfitting, yaitu terlalu menyesuaikan diri dengan data training tetapi kurang baik saat menghadapi pesan baru.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# TODO: buatkan fungsi untuk menghitung metriks klasifikasi

...

In [ ]:
# TODO: hitung metriks model pada training set

...

Pada classification, beberapa model juga bisa menghasilkan probability atau peluang untuk setiap kelas. Pada contoh ini, `spam_probability` menunjukkan seberapa besar peluang sebuah pesan dianggap sebagai `spam` oleh model.

`predict_proba()` menghasilkan probability untuk setiap kelas yang dipelajari model. Karena kita ingin mengambil peluang kelas `spam`, kita cari posisi kelas `spam` dari `model.classes_`, lalu mengambil probability pada kolom tersebut. Jika nilainya semakin mendekati 1, model semakin yakin pesan tersebut adalah spam.


In [ ]:
# TODO: lakukan prediksi pada test set

TAMPILKAN_PREDIKSI = 10

...

spam_class_index = list(model.classes_).index("spam")
y_pred_proba = model.predict_proba(X_test_tfidf)[:, spam_class_index]

prediction_preview = pd.DataFrame({
    "message": X_test.iloc[:TAMPILKAN_PREDIKSI].values,
    "actual": y_test.iloc[:TAMPILKAN_PREDIKSI].values,
    "predicted": y_pred[:TAMPILKAN_PREDIKSI],
    "spam_probability": y_pred_proba[:TAMPILKAN_PREDIKSI].round(4)
})

prediction_preview

Setelah melihat performa pada training set, evaluasi juga perlu dilakukan pada test set.

Test set berisi pesan yang tidak digunakan saat training, sehingga hasil evaluasi ini lebih menggambarkan kemampuan model pada data baru. Bandingkan metrik training dan test untuk melihat apakah model cukup general atau mulai menunjukkan tanda overfitting.

In [ ]:
# TODO: hitung metriks model pada test set

...

Confusion matrix menunjukkan detail kesalahan model. Untuk kasus spam detection, false positive berarti SMS biasa salah dianggap spam, sedangkan false negative berarti SMS spam lolos sebagai SMS biasa.


In [ ]:
from sklearn.metrics import confusion_matrix

# TODO: hitung confusion matrix

cm = ...

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["ham", "spam"],
    yticklabels=["ham", "spam"]
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

## Membuat Pipeline versi ColumnTransformer

Preprocessing sebelumnya dibuat secara manual agar setiap langkah lebih mudah dipahami. Namun dalam workflow machine learning yang lebih rapi, preprocessing dan model sebaiknya dibungkus dalam pipeline.

Karena dataset classification ini hanya memakai satu kolom teks, `ColumnTransformer` digunakan untuk menerapkan pipeline teks pada kolom `message`. Dengan cara ini, model bisa menerima data mentah dan seluruh proses cleaning, TF-IDF, dan training berjalan dalam satu alur.


In [ ]:
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

In [ ]:
class TextCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]

        return pd.Series(X).apply(clean_text).to_numpy()


class SpamClassificationModel:
    def __init__(self, model=None, vectorizer=None):
        if model is None:
            model = LogisticRegression(max_iter=1000, random_state=42)

        if vectorizer is None:
            vectorizer = TfidfVectorizer(stop_words="english", max_features=3000)

        self.model = model
        self.vectorizer = vectorizer
        self.text_feature = "message"

        text_pipeline = Pipeline([
            ("cleaner", TextCleaner()),
            ("tfidf", self.vectorizer)
        ])

        preprocessor = ColumnTransformer([
            ("text", text_pipeline, self.text_feature)
        ])

        self.pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", self.model)
        ])

    def train(self, X_train, y_train):
        self.pipeline.fit(X_train, y_train)
        return self

    def evaluate(self, X, y):
        y_pred = self.pipeline.predict(X)

        return calculate_metrics(y, y_pred)

    def predict(self, X):
        return self.pipeline.predict(X)

    def predict_proba(self, X):
        if hasattr(self.pipeline, "predict_proba"):
            return self.pipeline.predict_proba(X)

        raise AttributeError("Model ini tidak memiliki method predict_proba().")

    def tune(
        self,
        X_train,
        y_train,
        param_grid,
        cv=5,
        scoring=None,
        n_jobs=-1
    ):
        if scoring is None:
            scoring = make_scorer(f1_score, pos_label="spam")

        search = GridSearchCV(
            estimator=self.pipeline,
            param_grid=param_grid,
            cv=cv,
            scoring=scoring,
            n_jobs=n_jobs
        )

        search.fit(X_train, y_train)
        self.pipeline = search.best_estimator_

        return {
            "best_params": search.best_params_,
            "best_score": search.best_score_
        }

Pada versi pipeline, kita mulai lagi dari pesan mentah agar seluruh preprocessing dilakukan di dalam pipeline. Ini lebih aman untuk eksperimen karena cleaning dan TF-IDF selalu ikut terbawa saat training, evaluation, tuning, dan inference.


In [ ]:
df_pipeline = pd.read_csv("dataset/spam.csv", encoding="latin-1")

# gabungkan potongan pesan dari kolom Unnamed jika ada
message_columns = ["v2", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]

df_pipeline["message"] = (
    df_pipeline[message_columns]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ambil kolom yang dibutuhkan
df_pipeline = df_pipeline[["v1", "message"]]
df_pipeline.columns = ["label", "message"]

# hapus duplikat
df_pipeline = df_pipeline.drop_duplicates().reset_index(drop=True)

In [ ]:
X_raw = df_pipeline[["message"]]
y = df_pipeline["label"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=UKURAN_TEST_SET,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

USE_MODEL = LogisticRegression(max_iter=1000, random_state=42)

spam_model = SpamClassificationModel(model=USE_MODEL)

spam_model.train(X_train_raw, y_train)

print(f"Metriks evaluasi pada training set: {spam_model.evaluate(X_train_raw, y_train)}")
print(f"Metriks evaluasi pada test set: {spam_model.evaluate(X_test_raw, y_test)}")

## Melakukan Prediksi dengan Pesan Baru

Setelah pipeline dilatih, kita bisa menggunakannya untuk memprediksi pesan baru. Ubah isi `new_messages` untuk mencoba skenario yang berbeda, misalnya pesan promosi, pesan hadiah palsu, atau pesan percakapan biasa.


In [ ]:
# NOTE: ubah input untuk mencoba prediksi

new_messages = pd.DataFrame({
    "message": [
        "Congratulations, you won a free ticket! Claim your prize now",
        "Can we meet at the library after class?",
        "URGENT! Your account has been selected for a cash reward",
        "Please bring your notebook for tomorrow workshop"
    ]
})

predicted_label = spam_model.predict(new_messages)

spam_class_index = list(spam_model.pipeline.classes_).index("spam")
predicted_proba = spam_model.predict_proba(new_messages)[:, spam_class_index]

inference_result = new_messages.copy()
inference_result["prediction_label"] = predicted_label
inference_result["spam_probability"] = predicted_proba.round(4)

inference_result

## Eksperimen dengan Model lain dan Hyperparameter Tuning

Dalam machine learning, kita biasanya tidak berhenti pada satu model saja. Model yang berbeda bisa menangkap pola data dengan cara yang berbeda, sehingga kita perlu membandingkan beberapa pilihan.

Pada bagian ini, kita mencoba beberapa model classification dan melakukan hyperparameter tuning dengan `GridSearchCV`. Karena target masih berbentuk teks `ham` dan `spam`, kita membuat scorer khusus agar F1-score dihitung dengan `spam` sebagai positive class.


In [ ]:
from sklearn.svm import LinearSVC

spam_f1_scorer = make_scorer(f1_score, pos_label="spam")

USE_MODEL = LinearSVC(random_state=42, max_iter=5000)

spam_model = SpamClassificationModel(model=USE_MODEL)

param_grid = {
    "preprocessor__text__tfidf__max_features": [1000, 3000, 5000],
    "model__C": [0.1, 1, 10]
}

result = spam_model.tune(
    X_train_raw,
    y_train,
    param_grid=param_grid,
    scoring=spam_f1_scorer,
    cv=5
)

print(result)
print()

print(f"Metriks evaluasi pada training set: {spam_model.evaluate(X_train_raw, y_train)}")
print(f"Metriks evaluasi pada test set: {spam_model.evaluate(X_test_raw, y_test)}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier

USE_MODEL = DecisionTreeClassifier(random_state=42)

spam_model = SpamClassificationModel(model=USE_MODEL)

param_grid = {
    "preprocessor__text__tfidf__max_features": [1000, 3000, 5000],
    "model__max_depth": [10, 20, 30, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5]
}

result = spam_model.tune(
    X_train_raw,
    y_train,
    param_grid=param_grid,
    scoring=spam_f1_scorer,
    cv=5
)

print(result)
print()

print(f"Metriks evaluasi pada training set: {spam_model.evaluate(X_train_raw, y_train)}")
print(f"Metriks evaluasi pada test set: {spam_model.evaluate(X_test_raw, y_test)}")

In [ ]:
print("Congratulations, Classification is Finished!")